In [1]:
!pip install -q transformers sentencepiece gradio accelerate torch

In [2]:
# 2) Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr
import torch


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL = "facebook/nllb-200-distilled-600M"   # Public, works without login

tokenizer = AutoTokenizer.from_pretrained(MODEL , use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to("cuda")

# Bangla target language code for NLLB
BN_LANG = "ben_Beng"
forced_bos_token_id = tokenizer.convert_tokens_to_ids(BN_LANG)

print("NLLB Model Loaded Successfully!")
print("Bangla BOS Token ID:", forced_bos_token_id)


NLLB Model Loaded Successfully!
Bangla BOS Token ID: 256026


In [10]:
def translate_en_to_bn(text, max_len=200):
    if not text or text.strip() == "":
        return ""

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,  # IMPORTANT: Forces Bangla output
        max_length=max_len,
        num_beams=4,
        no_repeat_ngram_size=3
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [11]:
import gradio as gr

def interface_fn(text, max_len):
    return translate_en_to_bn(text, max_len)

demo = gr.Interface(
    fn=interface_fn,
    inputs=[
        gr.Textbox(lines=4, label="English Text (Input)"),
        gr.Slider(50, 300, value=200, step=10, label="Max Output Length")
    ],
    outputs=gr.Textbox(lines=5, label="বাংলা অনুবাদ (Bangla Translation)"),
    title="English → বাংলা Translator (NLLB-200)",
    description="High-quality machine translation using Meta's NLLB-200 distilled model."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://67ff015af0523799e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
